# R19-H188 - The Document-Grounded Wide Probe Set

**Author**: KGF experiment executor  
**Approach**: derive >=100 golds from SOURCE DOCUMENT text spans (not graph props), double-run the render harness (wide vs the 33-gold hand-curated set), and adjudicate the three H188 clauses.

H186 refuted its own validity: graph-derived golds scored 99.2% because they echo the render surface. Here every gold is a document text span - feature names and unit-bearing spec values lifted verbatim from the parsed corpus texts (`reports/parser-round-cache/`), phrasing independent of what extraction wrote into the graph. Product identity comes from the graph (metadata only); the evidence string is document wording.

**Bars** - (a) recall@8 in 55-75% AND recall@16 minus recall@8 >= 10 pts; (b) bootstrap CI half-width < 2.5 pts; (c) value-type filter keeps clean fraction >= 85%. Refutation (document golds ALSO saturate) is an explicitly valuable outcome.

## Setup - harness (graph pull, render, Titan query embeddings)

CPU only, no LLM. Graph is READ-ONLY neo4j2. Query embeddings via Bedrock Titan, cached for deterministic re-execution. The vector index is approximate (HNSW), so retrieval uses a generous `top_k=48` truncated to k=8/16 - requesting exactly k understates recall@16.

In [ ]:
import os, re, json, math, pickle, hashlib, random, datetime
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
from rich import print as rprint

from knowledge_graph_foundry import load_settings, Foundry
from knowledge_graph_foundry.extraction import generate_embeddings
from knowledge_graph_foundry.graph.graphrag import vector_query
from knowledge_graph_foundry.models import Entity

ROOT = Path("..")
BASE_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # baseline, READ-ONLY
settings = load_settings(ROOT / "config.yml")
VEC = settings.graphrag.vector_index_name
RETRIEVE_K = 48                                          # HNSW headroom, truncated to k=8/16
np.random.seed(42)

def _norm(s): return re.sub(r"\s+", " ", (s or "").casefold())
def value_tokens(t): return re.findall(r"[\w.\-/]*\d[\w.\-/]*", t or "")
_UN = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"
def present(gold, ctx):                                  # H34/H61 fuzzy evidence matcher (verbatim)
    ng = _norm(gold)
    if ng in ctx: return True
    sq = re.sub(r"[\s,()]", "", ctx); sk = re.sub(r"[\s,()]", "", re.sub(_UN, "", ng))
    if any(c.isdigit() for c in sk) and len(sk) >= 5 and sk in sq: return True
    tok = value_tokens(gold)
    if tok:
        hit = sum(1 for t in tok if _norm(t) in ctx or re.sub(r"[\s,()]", "", _norm(t)) in sq)
        return hit >= max(1, len(tok)//2 + (len(tok) % 2))
    w = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng)); cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx))
    return bool(w) and len(w & cw)/len(w) >= 0.6
def is_shortnum(g):                                       # fuzzy-matcher degeneracy class
    t = value_tokens(g); return bool(t) and sum(len(re.sub(r"[^\d]", "", x)) for x in t) <= 3
def unit(v):
    a = np.array(v, dtype=np.float32); n = np.linalg.norm(a); return a/n if n else a

# ---- graph pull (READ-ONLY) ----
d = GraphDatabase.driver(BASE_URI, auth=("neo4j", "kgfoundry"))
with d.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    prod_rows = s.run("MATCH (e:Entity) WHERE any(l IN labels(e) WHERE l IN ['CPAPDevice','ProductModel']) "
                      "AND e.name IS NOT NULL RETURN DISTINCT e.name AS n, e.source_documents AS sd").data()
    docmap = {r["id"]: r["nm"] for r in s.run("MATCH (dd:KGFDocument) RETURN dd.id AS id, dd.name AS nm").data()}
d.close()
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))

def render_nodes(ids):                                    # baseline render surface (name/desc/props/relations)
    blocks = []
    for nid in ids:
        r = node.get(nid)
        if r is None: continue
        spec = {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
        rl = rels_by.get(nid, [])[:15]
        blocks.append(f"## {r['name']} ({', '.join(r['types'])})\n{r['description'] or ''}\n"
                      f"Properties: {json.dumps(spec, default=str)}\nRelations: " +
                      "; ".join(f"{t} -> {names.get(b, '')}" for t, b in rl))
    return _norm("\n".join(blocks))

# ---- Titan query-embedding cache ----
QC = Path(".wide_probes_h188_qcache.pkl")
qcache = pickle.load(open(QC, "rb")) if QC.exists() else {}
def qemb(q):
    k = hashlib.md5(q.encode()).hexdigest()
    if k not in qcache:
        qcache[k] = generate_embeddings([Entity.create(q[:80], types=["Query"], description=q)],
                                        settings.embeddings)[0].embedding
    return qcache[k]

# ---- parsed corpus texts ----
PC = ROOT / "reports/parser-round-cache"
texts = {pr: json.loads((PC / f"text_{pr}.json").read_text()) for pr in ["docling","pymupdf4llm","pdfplumber","pypdf"]}
DOCS = list(texts["docling"].keys())
docnorm = {dn: _norm(" ".join(texts[pr].get(dn, "") for pr in texts)) for dn in DOCS}
raw = {dn: "\n".join(texts[pr].get(dn, "") for pr in ["pdfplumber","pymupdf4llm","docling"]) for dn in DOCS}
doclines = {dn: [l for pr in ["pdfplumber","pymupdf4llm","docling"] for l in texts[pr].get(dn, "").split("\n")] for dn in DOCS}
rprint(f"[green]harness ready[/green]  entities [yellow]{len(node)}[/yellow]  docs [yellow]{len(DOCS)}[/yellow]  "
       f"vector index [yellow]{VEC}[/yellow]  retrieve_k [yellow]{RETRIEVE_K}[/yellow]")

## Derivation - document-grounded golds

Two deterministic rules, both sourced from document wording:

- **`doc_feature_*`** - enumerable feature/mode names (AutoSet, C-Flex, heated tube, ...) found in a product's document. Branded features carry a brand guard (a ResMed feature attaches only to a ResMed-named product) to kill cross-brand errors; inside dense feature-matrix catalogues attribution is same-line only (avoids transposed-matrix row bleed); generic features are restricted to brand-known devices
- **`doc_spec_proximity`** - unit-bearing spec values (weight kg, pressure cmH2O, sound dB, ...) within 140 chars of a product-name mention, regex-extracted from the document, never from graph props

Every gold is verified verbatim-present via the `present()` matcher. A value-type filter caps the degenerate short-numeric class so the clean fraction stays >= 85%.

In [ ]:
random.seed(11)
prods = []
for r in prod_rows:
    nm = r["n"]
    if len(nm) < 5: continue
    ds = [docmap.get(x) for x in (r["sd"] or []) if docmap.get(x) in docnorm and _norm(nm) in docnorm[docmap.get(x)]]
    if ds: prods.append((nm, ds))
docprod = Counter()
for nm, ds in prods:
    for dn in ds: docprod[dn] += 1
FOCUSED = {dn for dn, c in docprod.items() if c <= 6}     # reliable proximity attribution
MATRIX  = {dn for dn, c in docprod.items() if c > 11}     # dense feature matrices
RESMED  = ("airsense","aircurve","airmini","airstart","lumis","resmed","air10","air11","s9")
PHILIPS = ("dreamstation","dreamwear","remstar","bipap","respironics","amara")
def brand(nm):
    n = _norm(nm)
    if any(t in n for t in RESMED): return "resmed"
    if any(t in n for t in PHILIPS): return "philips"
    return None
BRANDED = [("AutoSet","AutoSet","resmed"),("EPR","EPR","resmed"),("AutoRamp","AutoRamp","resmed"),
 ("SmartStart","SmartStart","resmed"),("Climate Control","Climate Control","resmed"),
 ("C-Flex+","C-Flex+","philips"),("C-Flex","C-Flex","philips"),("A-Flex","A-Flex","philips"),
 ("Bi-Flex","Bi-Flex","philips"),("System One","System One resistance control","philips"),
 ("Opti-Start","Opti-Start","philips"),("CPAP-Check","CPAP-Check","philips"),
 ("Auto-Trial","Auto-Trial","philips"),("EZ-Start","EZ-Start","philips"),("P-Flex","P-Flex","philips")]
GENERIC = [("heated tube","heated tube"),("heated humidif","heated humidifier"),("mask fit","mask fit check"),
 ("leak compensation","leak compensation"),("auto altitude","automatic altitude adjustment"),
 ("ramp time","adjustable ramp time"),("smartcard","SmartCard data storage"),
 ("bluetooth","Bluetooth connectivity"),("modem","integrated cellular modem")]
def near(low, starts, i, w): return any(abs(i - s) <= w for s in starts)

feat = []; fseen = set()
def addf(nm, label, dn, rule):
    k = (nm, label)
    if k in fseen or not present(label, docnorm[dn]): return
    fseen.add(k); feat.append(dict(product=nm, attr="supported feature", gold=label, doc=dn, rule=rule,
                                   q=f"Does the {nm} offer {label}?"))
for nm, ds in prods:
    br = brand(nm); nl = _norm(nm)
    for dn in ds:
        low = raw[dn].lower(); starts = [m.start() for m in re.finditer(re.escape(nl), low)]
        if not starts: continue
        lines = [l for l in doclines[dn] if nl in _norm(l)]
        for pat, label, reqbr in BRANDED:
            if br != reqbr: continue
            if any(pat.lower() in _norm(l) for l in lines): addf(nm, label, dn, "doc_feature_branded")
            elif dn not in MATRIX:
                for fm in re.finditer(re.escape(pat.lower()), low):
                    if near(low, starts, fm.start(), 180): addf(nm, label, dn, "doc_feature_branded"); break
        if br is not None:
            for pat, label in GENERIC:
                if any(pat.lower() in _norm(l) for l in lines): addf(nm, label, dn, "doc_feature_generic")
                elif dn in FOCUSED:
                    for fm in re.finditer(re.escape(pat.lower()), low):
                        if near(low, starts, fm.start(), 150): addf(nm, label, dn, "doc_feature_generic"); break

ATTR = [("weight","weight",r"\b\d[\d.,]*\s?(?:kg|lbs?|kilograms?|grams?)\b"),
 ("sound","sound level",r"\b\d[\d.,]*\s?dB\s?\(?A?\)?\b"),("noise","noise level",r"\b\d[\d.,]*\s?dB\s?\(?A?\)?\b"),
 ("pressure","operating pressure range",r"\b\d[\d.,]*\s?[-–]\s?\d[\d.,]*\s?(?:cm\s?H\s?2?\s?O|hpa)\b"),
 ("ramp","ramp time",r"\b\d[\d.,]*\s?(?:min(?:ute)?s?)\b"),
 ("humidif","humidifier water capacity",r"\b\d[\d.,]*\s?(?:ml|millilitres?)\b"),
 ("water","water chamber capacity",r"\b\d[\d.,]*\s?(?:ml|millilitres?)\b"),
 ("warrant","warranty period",r"\b\d\s?(?:years?|yr)\b"),
 ("dimension","dimensions",r"\b\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?(?:mm|cm)?\b"),
 ("power","power supply",r"\b\d[\d.,]*\s?(?:w|watts?)\b"),("frequency","operating frequency",r"\b\d[\d.,]*\s?hz\b")]
spec = []; sseen = set()
for nm, ds in prods:
    nl = _norm(nm)
    for dn in ds:
        txt = raw[dn]; low = txt.lower(); starts = [m.start() for m in re.finditer(re.escape(nl), low)]
        if not starts: continue
        for akw, lab, vre in ATTR:
            key = (nm, lab)
            if key in sseen: continue
            for am in re.finditer(re.escape(akw), low):
                ai = am.start()
                if not near(low, starts, ai, 140): continue
                vm = re.search(vre, txt[max(0, ai-15):ai+130], re.I)
                if not vm: continue
                val = re.sub(r"\s+", " ", vm.group(0)).strip()
                if len(val) < 2 or not re.search(r"\d", val) or not present(val, docnorm[dn]): continue
                sseen.add(key); spec.append(dict(product=nm, attr=lab, gold=val, doc=dn, rule="doc_spec_proximity",
                                                 q=f"What is the {lab} of the {nm}?")); break

for g in feat: g["short_numeric"] = False
for g in spec: g["short_numeric"] = is_shortnum(g["gold"])
allg = feat + spec
allg = [g for g in allg if not (g["rule"].startswith("doc_feature") and _norm(g["gold"]) in _norm(g["product"]))]  # drop trivial feature-in-name
clean_g = [g for g in allg if not g["short_numeric"]]; short_g = [g for g in allg if g["short_numeric"]]
cap = int(len(clean_g)/0.85 - len(clean_g))              # value-type filter: cap degenerate class -> clean >= 85%
random.shuffle(short_g)
wide = clean_g + short_g[:max(0, cap)]
random.shuffle(wide)
for i, g in enumerate(wide): g["id"] = f"W{i+1:03d}"
clean_frac = sum(1 for g in wide if not g["short_numeric"]) / len(wide)

art = [dict(id=g["id"], question=g["q"], gold_evidence=[g["gold"]], source_document=g["doc"],
            derivation_rule=g["rule"], product=g["product"], attribute=g["attr"],
            short_numeric=g["short_numeric"]) for g in wide]
outp = ROOT / "data/processed/probes-wide-h188.json"
outp.write_text(json.dumps(dict(meta=dict(
    n=len(wide), clean_fraction=clean_frac, date="2026-07-07",
    source="parsed source-document texts (docling/pymupdf4llm/pdfplumber); product identity from graph, evidence strings from document wording",
    verification="each gold verbatim-present in source_document via the H34 present() matcher"),
    probes=art), indent=2))
rprint(f"[bold cyan]wide set[/bold cyan] [yellow]{len(wide)}[/yellow] golds  clean fraction [yellow]{clean_frac:.1%}[/yellow] "
       f"[dim](bar >=85%)[/dim]  by rule {dict(Counter(g['rule'] for g in wide))}")
rprint(f"  [dim]saved {outp}[/dim]")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), constrained_layout=True)
rc = Counter(g["rule"] for g in wide)
ax[0].bar(range(len(rc)), list(rc.values()), color="#4C78A8")
ax[0].set_xticks(range(len(rc))); ax[0].set_xticklabels([k.replace("doc_", "") for k in rc], rotation=20, ha="right")
ax[0].set_title("golds by derivation rule"); ax[0].set_ylabel("count")
cc = [sum(1 for g in wide if not g["short_numeric"]), sum(1 for g in wide if g["short_numeric"])]
ax[1].bar(["clean (feature / unit value)", "degenerate short-numeric"], cc, color=["#59A14F", "#E15759"])
ax[1].axhline(len(wide)*0.15, ls="--", c="gray", lw=1); ax[1].set_title(f"value-type filter - clean {clean_frac:.1%}")
plt.show()

## Render double-run - wide (document-grounded) vs small (33-gold hand-curated)

`score_set` retrieves each question's seeds via the Titan-embedded vector query (`top_k=48`), renders the top-8 and top-16 nodes, and matches the gold with `present()`. Recall@8, recall@16, the k-lever, and a per-rule breakdown are computed for the wide set; the small hand-curated set is the difficulty reference. A 5000-sample bootstrap gives the CI half-width.

In [ ]:
import yaml
def score_hits(items):
    h8 = []; h16 = []
    with Foundry(settings) as f:
        for it in items:
            qv = unit(qemb(it["q"])); res = vector_query(f.driver, qv, VEC, top_k=RETRIEVE_K)
            seeds = [x["id"] for x in res if x["id"] in node]
            h8.append(present(it["gold"], render_nodes(seeds[:8])))
            h16.append(present(it["gold"], render_nodes(seeds[:16])))
    return np.array(h8, dtype=float), np.array(h16, dtype=float)

wh8, wh16 = score_hits(wide)
probes = yaml.safe_load((ROOT / "tests/probes/cpap-probe-set.yml").read_text())
small = [dict(q=p["question"], gold=g) for p in probes if p.get("gold_evidence") for g in p["gold_evidence"]]
sh8, sh16 = score_hits(small)
pickle.dump(qcache, open(QC, "wb"))

N, M = len(wide), len(small)
wr8, wr16, sr8, sr16 = wh8.mean(), wh16.mean(), sh8.mean(), sh16.mean()
rng = np.random.default_rng(42)
def halfwidth(h):
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(5000)]
    return (np.percentile(b, 97.5) - np.percentile(b, 2.5)) / 2 * 100
w8hw, w16hw = halfwidth(wh8), halfwidth(wh16)

perrule = {}
for g, h8, h16 in zip(wide, wh8, wh16):
    r = perrule.setdefault(g["rule"], [0, 0.0, 0.0]); r[0] += 1; r[1] += h8; r[2] += h16

rprint(f"[bold cyan]WIDE[/bold cyan]  N=[yellow]{N}[/yellow] recall@8=[yellow]{wr8:.3f}[/yellow] recall@16=[yellow]{wr16:.3f}[/yellow] "
       f"lever=[yellow]{(wr16-wr8)*100:+.1f}[/yellow]pts  CI half-width@8=[yellow]{w8hw:.2f}[/yellow]pts")
rprint(f"[bold cyan]SMALL[/bold cyan] M=[yellow]{M}[/yellow] recall@8=[yellow]{sr8:.3f}[/yellow] recall@16=[yellow]{sr16:.3f}[/yellow] "
       f"lever=[yellow]{(sr16-sr8)*100:+.1f}[/yellow]pts")
for rule, (n, h8, h16) in sorted(perrule.items()):
    rprint(f"  [dim]{rule:22} N={n:3} recall@8={h8/n:.3f} recall@16={h16/n:.3f} lever={(h16-h8)/n*100:+.1f}pts[/dim]")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
x = np.arange(2); wdt = 0.35
ax[0].bar(x - wdt/2, [wr8, wr16], wdt, label="wide (doc-grounded)", color="#4C78A8")
ax[0].bar(x + wdt/2, [sr8, sr16], wdt, label="small (hand-curated)", color="#F28E2B")
ax[0].axhspan(0.55, 0.75, color="green", alpha=0.08); ax[0].set_xticks(x); ax[0].set_xticklabels(["recall@8", "recall@16"])
ax[0].set_ylim(0, 1); ax[0].set_title("wide vs small - 55-75% band shaded"); ax[0].legend(fontsize=8)
rules = sorted(perrule); r8 = [perrule[r][1]/perrule[r][0] for r in rules]; r16 = [perrule[r][2]/perrule[r][0] for r in rules]
xx = np.arange(len(rules))
ax[1].bar(xx - wdt/2, r8, wdt, label="@8", color="#59A14F"); ax[1].bar(xx + wdt/2, r16, wdt, label="@16", color="#B6D7A8")
ax[1].set_xticks(xx); ax[1].set_xticklabels([r.replace("doc_", "") for r in rules], rotation=20, ha="right")
ax[1].set_ylim(0, 1); ax[1].set_title("wide recall by rule"); ax[1].legend(fontsize=8)
plt.show()

## Verdict

In [ ]:
clause_a = (0.55 <= wr8 <= 0.75) and ((wr16 - wr8) * 100 >= 10)
clause_b = w8hw < 2.5
clause_c = clean_frac >= 0.85
saturated = wr8 > 0.90
verdict = "CONFIRMED" if (clause_a and clause_b and clause_c) else "REFUTED"
report = dict(
    hypothesis="R19-H188", date=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    n_wide=N, n_small=M, clean_fraction=float(clean_frac),
    by_rule={r: dict(n=v[0], recall8=v[1]/v[0], recall16=v[2]/v[0]) for r, v in perrule.items()},
    wide_recall8=float(wr8), wide_recall16=float(wr16), wide_lever_pts=float((wr16-wr8)*100),
    small_recall8=float(sr8), small_recall16=float(sr16), small_lever_pts=float((sr16-sr8)*100),
    ci_halfwidth8_pts=float(w8hw), ci_halfwidth16_pts=float(w16hw),
    clause_a_band_and_lever=bool(clause_a), clause_b_ci_under_2p5=bool(clause_b),
    clause_c_clean_over_85=bool(clause_c), saturated_over_90=bool(saturated), verdict=verdict)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
rp = ROOT / f"reports/wide-probes-h188-{stamp}.json"
rp.write_text(json.dumps(report, indent=2))
rprint(f"clause a (recall@8 55-75% AND lever>=10pts): [{'green' if clause_a else 'red'}]{clause_a}[/]  "
       f"[dim]recall@8={wr8:.1%} lever={(wr16-wr8)*100:+.1f}pts[/dim]")
rprint(f"clause b (CI half-width < 2.5pts):            [{'green' if clause_b else 'red'}]{clause_b}[/]  [dim]{w8hw:.2f}pts[/dim]")
rprint(f"clause c (clean fraction >= 85%):             [{'green' if clause_c else 'red'}]{clause_c}[/]  [dim]{clean_frac:.1%}[/dim]")
rprint(f"[bold]VERDICT: {verdict}[/bold]  "
       f"[dim]document-grounded wide golds {'SATURATE' if saturated else 'do not saturate'} at recall@8={wr8:.1%}; "
       f"small hand-curated set stays hard ({sr8:.1%}, lever {(sr16-sr8)*100:+.1f}pts)[/dim]")
rprint(f"  [dim]saved {rp}[/dim]")